# ResNet18 Pretrained Experiment

This notebook trains and evaluates an ImageNet-pretrained ResNet18 model on the selected iNaturalist subset.

Goal:
- Use transfer learning with ImageNet-pretrained weights.
- Compare against the Simple CNN baseline trained from scratch.
- Save training history, metrics, predictions, figures, and model weights for the final report.

## 1. Imports and Project Paths

In [ ]:
# Standard library imports
from pathlib import Path
from datetime import datetime
import sys

# Third-party imports
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# PROJECT_DIR is the parent folder of this notebook folder.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"

# Add src/ to Python's import path so this notebook can use our reusable code.
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

TRAIN_DIR = PROJECT_DIR / "select_train_mini"
VAL_DIR = PROJECT_DIR / "select_val_mini"
TRAIN_JSON = TRAIN_DIR / "selected_train_mini.json"
VAL_JSON = VAL_DIR / "selected_val_mini.json"

RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"
FIGURES_DIR = PROJECT_DIR / "figures"

RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Training folder exists:", TRAIN_DIR.exists())
print("Validation folder exists:", VAL_DIR.exists())
print("Training JSON exists:", TRAIN_JSON.exists())
print("Validation JSON exists:", VAL_JSON.exists())

## 2. Import Reusable Project Code

In [ ]:
# Reusable project helpers
from src.data_utils import load_selected_metadata, create_dataloaders, count_images_by_class
from src.resnet18_pretrained import build_resnet18_pretrained
from src.train_utils import train_model
from src.eval_utils import evaluate_model
from src.plot_utils import plot_training_history
from src.utils import set_seed, get_device, save_model_weights, append_experiment_log

## 3. Reproducibility and Device Check

In [ ]:
# A fixed seed makes random operations more consistent between runs.
SEED = 42
set_seed(SEED)

# cuDNN benchmark can speed up training when image sizes are fixed.
torch.backends.cudnn.benchmark = True

device = get_device()

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Load Metadata and Check Dataset Size

In [ ]:
# Load selected JSON metadata for reproducibility and report tables.
train_meta, train_categories, train_images, train_annotations = load_selected_metadata(TRAIN_JSON)
val_meta, val_categories, val_images, val_annotations = load_selected_metadata(VAL_JSON)

train_counts = count_images_by_class(TRAIN_DIR)
val_counts = count_images_by_class(VAL_DIR)

summary = train_counts.merge(
    val_counts,
    on="class_name",
    suffixes=("_train", "_val"),
)

class_info = train_categories[[
    "category_id", "name", "common_name", "kingdom", "family", "genus", "image_dir_name"
]].rename(columns={"image_dir_name": "class_name", "name": "species_name"})

summary = class_info.merge(summary, on="class_name", how="left")

print("Number of classes:", len(summary))
print("Total train images:", summary["image_count_train"].sum())
print("Total validation images:", summary["image_count_val"].sum())

display(summary.head())
display(summary[["image_count_train", "image_count_val"]].describe())

## 5. Create DataLoaders

In [ ]:
# These settings can be adjusted for speed or accuracy.
# For RTX 4060, batch size 64 is usually a reasonable starting point.
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 0  # Keep 0 for Windows/Jupyter stability; try 2 later if stable.

train_dataset, val_dataset, train_loader, val_loader = create_dataloaders(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

num_classes = len(train_dataset.classes)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Number of classes:", num_classes)
print("First class:", train_dataset.classes[0])

## 6. Build ResNet18 Pretrained Model

In [ ]:
# This uses ImageNet-pretrained weights and replaces the final layer with 900 species classes.
# freeze_backbone=False means we fine-tune the whole model.
resnet18_pretrained = build_resnet18_pretrained(
    num_classes=num_classes,
    freeze_backbone=False,
).to(device)

print(resnet18_pretrained.fc)
print("Number of output classes:", num_classes)

## 7. Model Smoke Test

In [ ]:
# Check that one batch can pass through the model before training.
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = resnet18_pretrained(images)

print("Input batch shape:", images.shape)
print("Output batch shape:", outputs.shape)
print("Label batch shape:", labels.shape)

assert outputs.shape[0] == images.shape[0]
assert outputs.shape[1] == num_classes

## 8. Train ResNet18 Pretrained

In [ ]:
# CrossEntropyLoss is suitable for multi-class classification.
criterion = nn.CrossEntropyLoss()

# A smaller learning rate is safer for fine-tuning pretrained models.
optimizer = optim.AdamW(
    resnet18_pretrained.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

# Start with 3 epochs to check speed and learning behaviour.
NUM_EPOCHS_RESNET18 = 3

resnet18_pretrained, resnet18_history = train_model(
    model=resnet18_pretrained,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=NUM_EPOCHS_RESNET18,
    model_name="ResNet18_pretrained",
)

display(resnet18_history)

## 9. Plot and Save Training Curves

In [ ]:
# Training curves are required for explaining training dynamics in the report.
curve_path = FIGURES_DIR / "resnet18_pretrained_training_curves.png"

plot_training_history(
    resnet18_history,
    title="ResNet18 Pretrained",
    save_path=curve_path,
)

## 10. Save Model Weights and Training History

In [ ]:
# Save local model weights for later evaluation or demo use.
# Do not include trained .pth files in the final code ZIP submission.
model_path = MODELS_DIR / "resnet18_pretrained_best.pth"
history_path = RESULTS_DIR / "resnet18_pretrained_history.csv"

save_model_weights(resnet18_pretrained, model_path)
resnet18_history.to_csv(history_path, index=False)

print("Training history saved to:", history_path)

## 11. Evaluate ResNet18 Pretrained

In [ ]:
# Evaluate on the validation set using report-friendly metrics.
resnet18_metrics, resnet18_predictions = evaluate_model(
    model=resnet18_pretrained,
    dataloader=val_loader,
    device=device,
    class_names=train_dataset.classes,
)

resnet18_metrics_df = pd.DataFrame([{
    "model": "ResNet18_pretrained",
    **resnet18_metrics,
}])

display(resnet18_metrics_df)
display(resnet18_predictions.head())

## 12. Save Evaluation Results

In [ ]:
# Save metrics and per-image predictions for tables and later error analysis.
metrics_path = RESULTS_DIR / "resnet18_pretrained_metrics.csv"
predictions_path = RESULTS_DIR / "resnet18_pretrained_predictions.csv"

resnet18_metrics_df.to_csv(metrics_path, index=False)
resnet18_predictions.to_csv(predictions_path, index=False)

print("Metrics saved to:", metrics_path)
print("Predictions saved to:", predictions_path)

## 13. Append Experiment Log

In [ ]:
# Keep one compact CSV table for comparing all experiments.
experiment_log = {
    "experiment_name": "ResNet18_pretrained_initial_test",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model_type": "ResNet18",
    "initialisation": "ImageNet pretrained",
    "pretrained": True,
    "freeze_backbone": False,
    "num_classes": num_classes,
    "num_train_images": len(train_dataset),
    "num_val_images": len(val_dataset),
    "input_image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS_RESNET18,
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "loss_function": "CrossEntropyLoss",
    "augmentation": "Resize, RandomHorizontalFlip, RandomRotation",
    "device": str(device),
    "best_train_acc": resnet18_history["train_acc"].max(),
    "best_val_acc": resnet18_history["val_acc"].max(),
    "final_train_acc": resnet18_history["train_acc"].iloc[-1],
    "final_val_acc": resnet18_history["val_acc"].iloc[-1],
    "final_train_loss": resnet18_history["train_loss"].iloc[-1],
    "final_val_loss": resnet18_history["val_loss"].iloc[-1],
    "top1_accuracy": resnet18_metrics["top1_accuracy"],
    "top5_accuracy": resnet18_metrics["top5_accuracy"],
    "macro_precision": resnet18_metrics["macro_precision"],
    "macro_recall": resnet18_metrics["macro_recall"],
    "macro_f1": resnet18_metrics["macro_f1"],
    "total_training_time_sec": resnet18_history["epoch_time_sec"].sum(),
    "total_inference_time_sec": resnet18_metrics["total_inference_time_sec"],
    "notes": "Initial transfer learning experiment using ImageNet-pretrained ResNet18."
}

experiment_log_path = RESULTS_DIR / "experiment_log.csv"
updated_experiment_log = append_experiment_log(experiment_log_path, experiment_log)

display(updated_experiment_log.tail())